# V3 s01 三臂整链观察 — 用户一次 Run all
预选s01，P0／旧／新、两钥，O33 crop104与delete13 seam52。一次CPU max-t构造；VAE1、D181×3、E181×3+E45×144=147、FFT147、读取294，codec编码/探测/解码各9；源/DiT/旧梯度0。A、原净32位及逐g/channel能量/峰值预算不变，无加幅/扫描/放rho/自动重试。
Notebook直接从RICHAAARC/SC-SSTW的dev/local-joint-state-payload-v1分支获取可用源码，Run all执行上述固定任务，无需手动上传ZIP。也支持已放置的可编辑SOURCE_ROOT目录。没有B64、大内嵌、hash/manifest/版本/Git-clean门禁。
E181三次仅full MP4 readback；RGB8可保存/量质但不单独再编码读取。局部E45来自真实编辑104帧再次MP4/readback；此预算不能拆清VAE/codec/reset全部因果。
真实域始终MODEL_DOMAIN_UNVALIDATED，toy tau/rho只是固定诊断参考，不能认证/PASS或由s01重调。native label reserve7不等于整链容7错；C3/C4/C7仍全拒的历史缺口保留。历史4host CPU分母/两host未决不伪造为本包物理执行。
attempt/STARTED先保存，失败或中断消耗；同一Drive此固定输出后续Run all仅保存结果只读，不重新构造/物理调用。盲的完整矩阵/全部scores/H/c_hat/校正证据/来源及payload先保存回读，再读truth/P0配对量质。GPU/真实构造由用户执行，代理仅静态和stub核验。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Affine-Margin-Observation-V3/fixed-s01-once-v1')
def execution_artifacts(output):
    out=Path(output);found=[]
    for name in ('attempt.json','ledger.json','result.json','evaluation.json','plan.json','fixed_rows.json','progress.json','post_truth_access.json','message_post_rows.json','payload_group_pairs.json','payload_cell_pairs.json'):
        for suffix in ('','.tmp'):
            path=out/(name+suffix)
            if path.exists() or path.is_symlink():found.append(name+suffix)
    for name in ('raw','blind','payload','private','PN','construction','native','latents','media','quality','markers','receipts'):
        root=out/name
        if root.exists():found.extend(str(x.relative_to(out)) for x in root.rglob('*') if x.is_file() or x.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-affine-margin-observation-v3')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch','dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_affine_margin_observation_v3_run.py').is_file():
        raise FileNotFoundError('Available source lacks the V3 entry; inspect the actual source download or use editable SOURCE_ROOT: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
else:
    print('Existing attempt: source/setup/construct/model/physical skipped. Saved JSON read-only.')


固定一次执行。只处理真实依赖/导入故障，不绑定版本或GPU型号；构造/物理失败不重试。

In [ ]:

import subprocess, sys, time, os, signal
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','update'], 'refresh package metadata for missing FFmpeg')
        command(['apt-get','install','-y','ffmpeg'], 'install missing ffmpeg/ffprobe')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable after dependency installation; see '+str(LOG))
    ensure_imports('import numpy,torch,cvxpy,clarabel', ['numpy','torch','cvxpy','clarabel'], 'tensor and fixed solver dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE dependencies')
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_affine_margin_observation_v3_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress.get('counts',{}).get('RETURNED',0), '/', progress['planned'], 'counts', progress.get('counts',{}), 'active', progress.get('active'), flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        print('Worker group signalled. STARTED completion may remain unknown; attempt consumed and later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


只读保存结果；失败、未决和拒读保留。实际认证域仍未验证。

In [ ]:

for name in ('result.json','evaluation.json'):
    path=OUTPUT/name
    if not path.is_file():
        print('Unavailable',name);continue
    try: value=json.loads(path.read_text())
    except (OSError,ValueError) as error:
        print('Saved read error',name,str(error));continue
    if name=='result.json':
        print('Execution:',value.get('status'),'calls:',value.get('kinds_started'),'fixed:',value.get('fixed_denominators'))
    else:
        print('Domain:',value.get('model_domain'),'alignment:',value.get('alignment'),'full32:',value.get('full_group'),'issues:',value.get('issues'))
print('Raw/blind full matrices, all H/c_hat and corrected rows: raw/ blind/. Quality/temporal: quality/. Output:',OUTPUT)
print('No calibration/PASS: describe saved observations only. This notebook has not been run on GPU by the agent.')
